Its easier to pull from bigquery and store in googlesheets: https://docs.google.com/spreadsheets/d/1677AMAPbqeNjJaDbXZ86tGtinkLTMwfWbYrAZrc5f_M/edit?gid=1146765514#gid=1146765514

In [0]:
import requests
from datetime import datetime

BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

params = {
    "query": '"Jordan inflation"',
    "mode": "timelinevolraw",
    "format": "json",
    "startdatetime": "20240101000000",
    "enddatetime": "20241231235959"
}

headers = {
    "User-Agent": "info_env_jordan_research_project/1.0"
}

response = requests.get(BASE_URL, params=params, headers=headers, timeout=60)

print(response.status_code)
print(response.url)
print(response.text[:1000])

In [0]:
import requests
import time

BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

headers = {
    "User-Agent": "info_env_jordan_research_project/1.0"
}

params = {
    "query": '"Jordan inflation"',
    "mode": "timelinevol",
    "format": "json",
    "startdatetime": "20240101000000",
    "enddatetime": "20241231235959",
    "timelinesmooth": 7
}

response = requests.get(
    BASE_URL,
    params=params,
    headers=headers,
    timeout=60
)

print(response.status_code)
print(response.text[:500])

In [0]:
import requests
import pandas as pd
from datetime import datetime
import time

BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

search_terms = [
    "Jordan inflation",
    "Jordan protests",
    "Jordan unemployment",
    "Jordan economy",
    "Jordan fuel prices",
    "Jordan public sector strike"
]

start_date = "20200101000000"
end_date = datetime.now().strftime("%Y%m%d%H%M%S")

all_results = []


def get_gdelt_response(params, max_retries=5):
    for attempt in range(max_retries):
        response = requests.get(BASE_URL, params=params, timeout=30)

        if response.status_code == 200:
            return response

        if response.status_code == 429:
            wait_time = 10 * (attempt + 1)
            print(f"Rate limited. Waiting {wait_time} seconds before retrying...")
            time.sleep(wait_time)
            continue

        print(f"Request failed with status {response.status_code}: {response.text[:300]}")
        return None

    print("Max retries exceeded.")
    return None


for term in search_terms:
    params = {
        "query": f'"{term}"',
        "mode": "timelinevol",
        "format": "json",
        "startdatetime": start_date,
        "enddatetime": end_date,
        "timelinesmooth": 7
    }

    response = get_gdelt_response(params)

    if response is None:
        print(f"Skipping {term}")
        continue

    print(term, response.status_code)

    data = response.json()

    for row in data.get("timeline", []):
        all_results.append({
            "search_term": term,
            "date": row.get("date"),
            "article_volume": row.get("value")
        })

    time.sleep(5)

if not all_results:
    print("No data retrieved.")
else:
    gdelt_df = pd.DataFrame(all_results)
    display(gdelt_df)